# Citi Bike Project — Detailed Cheat Sheet

Every function pattern used in this project, organized by task, with a
mini-example for each. Skim the headers, jump to what you need.

## Loading & inspecting data

In [ ]:
df <- read.csv("file.csv")     # load a CSV into a data frame
head(df)                        # first 6 rows
str(df)                         # column names + types, fast way to spot problems
colSums(is.na(df))              # missing values per column
nrow(df); ncol(df)               # dimensions

## dplyr verbs (the ones this project uses)

In [ ]:
# filter(): keep rows matching a condition
df %>% filter(tripduration < 900)
df %>% filter(age < 80, gender == 1 | gender == 2)   # AND via comma, OR via |

# select(): keep only certain columns
df %>% select(start.station.longitude, start.station.latitude)

# mutate(): add or overwrite a column
df %>% mutate(age = 2020 - birth.year)
df %>% mutate(gender = as.factor(gender))            # cast a column's type

# group_by() + summarize(): one row per group, with aggregate stats
df %>% group_by(age) %>% summarize(mean_speed = mean(speed))
df %>% group_by(age, gender) %>% summarize(mean_speed = mean(speed))  # multiple keys

# group_by() + tally(): just counts per group (shortcut for summarize(n = n()))
df %>% group_by(age, gender) %>% tally()

# joins: combine two data frames on a shared key
full_join(a, b, by = c("colA" = "colB"))   # keep rows from both even if unmatched
inner_join(a, b, by = "date")               # keep only rows that match in both

# arrange(): sort
df %>% arrange(net_flow)          # ascending
df %>% arrange(desc(net_flow))    # descending

# rename(): change a column name
df %>% rename(station = start.station.name)

# across(): apply the same transformation to multiple columns
df %>% mutate(across(c(departures, arrivals), ~replace(., is.na(.), 0)))

## The pipe `%>%`

In [ ]:
# `x %>% f(y)` is the same as `f(x, y)` — it reads left-to-right as "do this,
# then this, then this," which is why dplyr chains stack so cleanly:

short_trips <- all_data %>%
  filter(tripduration < 900) %>%
  mutate(age = 2020 - birth.year) %>%
  mutate(speed = distance / tripduration)

## Distance & speed (geosphere)

In [ ]:
library(geosphere)

# distHaversine(p1, p2) wants two-column matrices/data frames of (lon, lat) —
# note LONGITUDE FIRST, then latitude. Returns distance in METERS.
starting_stations <- df %>% select(start.station.longitude, start.station.latitude)
ending_stations   <- df %>% select(end.station.longitude, end.station.latitude)

df <- df %>% mutate(distance = distHaversine(starting_stations, ending_stations))

# speed = distance (m) / tripduration (s) -> meters per second
df <- df %>% mutate(speed = distance / tripduration)

# Haversine = straight-line ("as the crow flies") distance. It UNDERESTIMATES
# real riding distance, which follows streets. A common fudge factor for
# urban grids is roughly x1.3-1.4.

## Date/time parsing (lubridate)

In [ ]:
library(lubridate)

df <- df %>% mutate(
  start_dt   = ymd_hms(starttime),         # parse "YYYY-MM-DD HH:MM:SS" strings
  hour       = hour(start_dt),             # 0-23
  weekday    = wday(start_dt, label = TRUE), # "Mon".."Sun" as a factor
  is_weekend = weekday %in% c("Sat", "Sun"),
  date       = as_date(start_dt)           # just the calendar date, for daily joins
)

# If your timestamp format is different, check it first:
head(df$starttime)
# common alternatives: mdy_hms(), dmy_hms(), or parse_date_time(x, orders = "...")

## ggplot2 building blocks

In [ ]:
library(ggplot2)

# Every ggplot has 3 parts: data, an aes() mapping, and one or more geoms.
ggplot(df, aes(x = age, y = mean_speed)) + geom_line()

# 2D heat map / density of points (great for lat/lon):
ggplot(df, aes(x = lon, y = lat)) + geom_bin2d(binwidth = c(0.001, 0.001))

# Line colored by a group:
ggplot(df, aes(x = age, y = mean_speed, color = gender)) + geom_line()

# Stacked bar / column chart:
ggplot(df, aes(x = age, y = n, fill = gender)) + geom_col()

# Horizontal bar chart (flip x/y after building it normally):
ggplot(df, aes(x = reorder(station, value), y = value)) + geom_col() + coord_flip()

# Scatter + trend line:
ggplot(df, aes(x = temp, y = trip_count)) + geom_point() + geom_smooth(method = "lm")

# Facets: small multiples split by a categorical column
ggplot(df, aes(x = hour, y = n)) + geom_line() + facet_wrap(~is_weekend)

# Titles, axis labels, centered title:
... + labs(title = "My title", x = "X axis", y = "Y axis") +
      theme(plot.title = element_text(hjust = 0.5))

# Custom legend labels for a discrete color/fill scale:
... + scale_color_discrete(name = "Gender", labels = c("Male Identifying", "Female Identifying"))
... + scale_fill_discrete(name = "Gender", labels = c("A", "B"))

## Categorical vs. continuous gotcha

In [ ]:
# If a column that's really a category (like gender: 0/1/2) is stored as a
# number, ggplot treats it as CONTINUOUS and gives you a gradient color scale
# instead of distinct colors/lines. Fix: cast it to a factor before plotting.

df <- df %>% mutate(gender = as.factor(gender))
# You'll see a coercion warning here -- that's expected, it's telling you the
# column's underlying type changed from integer to character/factor.

## Quick diagnostics / data-quality checks

In [ ]:
sum(df$tripduration <= 0)                       # non-positive durations
sum((2020 - df$birth.year) > 100)                 # implausible ages
sum(df$start.station.latitude == 0)               # bad geocoding
duplicated(df) %>% sum()                          # exact duplicate rows
range(df$tripduration, na.rm = TRUE)              # sanity-check min/max

## Common errors and what they mean

| Symptom | Likely cause | Fix |
|---|---|---|
| `could not find function "%>%"` | `dplyr` (or `magrittr`) not loaded | `library(dplyr)` |
| Legend shows a color gradient instead of distinct lines | grouping column is numeric, not a factor | `mutate(col = as.factor(col))` |
| `distHaversine()` gives huge/NaN distances | columns passed in wrong order or lon/lat swapped | pass `(longitude, latitude)` order, matching for both points |
| Plot renders empty / all one bin | `binwidth` too large for the coordinate scale | use `binwidth = c(0.001, 0.001)` for lon/lat in NYC |
| `ymd_hms()` returns `NA` | timestamp format doesn't match the parser | check `head()` of the raw string, use the matching `lubridate` parser |
| Notebook takes minutes to run each cell | using the full dataset, not the subset | filter early (`tripduration < 900`) or use the provided `_subset.csv` |

## Units cheat sheet

- `tripduration`: **seconds**
- `distHaversine()` output: **meters**
- `speed = distance / tripduration`: **meters per second** (multiply by ~2.237 for mph, ~3.6 for km/h)
- `birth.year` → `age = 2020 - birth.year` (dataset-specific: use the *data collection year*, not `Sys.Date()`)
- `gender` codes: `0` = unspecified, `1` = male-identifying, `2` = female-identifying